# 20 · M1.4b · Re-audit of the repaired CRT sampler on the real CPER features

**Status:** executes plan §5 and §8 of [docs/m14_sampler_experiment_plan.md](../docs/m14_sampler_experiment_plan.md) with the decisions notebook 19 fixed (`data/interim/m14/summary.pkl`):
- the eligible candidates and the chosen one;
- the block gap g;
- the audit threshold τ_A, and whether the audit was validated;
- whether power reached 0.80 at 0.05 nats.

None of these can change here, and there is no switching between candidates after this notebook.

**Decision rule (plan §8):**
- **suitable** = the chosen candidate is eligible, the audit is validated, and A ≤ τ_A on the real features;
- **exploratory Q0** = suitable, and power ≥ 0.80 at 0.05 nats on the real images;
- otherwise NEON Q0 development closes for this phase.

The real labels are read only in §4, and only if Q0 is authorised.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import warnings

import numpy as np
import pandas as pd

from visual_hygrometry import data, features, inference, samplers, suite

warnings.filterwarnings('ignore', message='Mean of empty slice')
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.precision', 3)
OFF = -7
SUMMARY = pd.read_pickle(data.ROOT / 'data' / 'interim' / 'm14' / 'summary.pkl')
U = pd.read_csv(data.ROOT / 'data' / 'interim' / 'm13_CPER.csv', parse_dates=['utc'])
VIS = [n for n in features.NAMES if not ((U.groupby('year').qc_sky_clip.median() > .2).any() and n.startswith('vis_sky'))]
loc = (U.utc + pd.Timedelta(hours=OFF)).dt.tz_localize(None)
R0 = pd.DataFrame({'year': U.year - 2020, 'doy': loc.dt.day_of_year, 'hour_local': loc.dt.hour + loc.dt.minute / 60,
                   'day': (loc.dt.floor('D') - pd.Timestamp('2020-01-01')).dt.days, 'temp_cond_c': U.t_asp})
R0 = pd.concat([R0, U[VIS], data.fourier(R0.hour_local, 24, 2, 'hour'), data.solar(R0.doy, R0.hour_local, suite.LAT, suite.LON, OFF)], axis=1)
yr, day, hour, temp, doy = (R0[c].to_numpy() for c in ('year', 'day', 'hour_local', 'temp_cond_c', 'doy'))
wk = yr * 14 + (doy - suite.DOY0) // 7
CRAW = np.c_[R0[['hour_local', 'temp_cond_c', 'doy', 'solar_elev_deg', 'solar_dayfrac']].to_numpy(), yr]
V0 = R0[VIS].to_numpy()
E0 = V0 - inference.project(CRAW, V0, wk)
SUMMARY

{'gap': 4,
 'eligible': [],
 'chosen': None,
 'tau_A': 0.5625,
 'audit_validated': 'inconclusive (no invalid pair)',
 'power_ok': False,
 'misspecified': {'M-a': 'unclassified',
  'M-b': 'unclassified',
  'M-c': 'unclassified',
  'M-d': 'unclassified',
  'M-e': 'unclassified'}}

## 1 · Held-out audit on the real features (every eligible candidate)

As in notebook 19 §5:
- For every season and each of its 4 contiguous blocks, the residual process is fitted on the season's other days farther than g days from the block.
- The block's residuals are simulated 100 times.
- The held-out diagnostics H1–H6 are compared with the 95 % band of the simulations.

A is the largest share of cells outside, over the families.

In [2]:
KIND = {'P': 'P', 'R': 'R'}
REAL = {k: samplers.holdout_audit(E0, day, hour, yr, hour, temp, KIND[k], SUMMARY['gap'], n_sim=100, seed=0) for k in SUMMARY['eligible']}
if REAL:
    sc = {k: samplers.audit_score(t) for k, t in REAL.items()}
    display(pd.DataFrame({k: s[0] for k, s in sc.items()}).rename_axis('family (share of cells outside)'))
    print({k: round(s[1], 3) for k, s in sc.items()}, f"τ_A = {SUMMARY['tau_A']:.3f}")
else: print('no eligible candidate in notebook 19: nothing to audit; NEON Q0 development closes (plan §8)')

no eligible candidate in notebook 19: nothing to audit; NEON Q0 development closes (plan §8)


In [3]:
if REAL:
    for k, t in REAL.items():
        print(f'candidate {k}: share outside by family and statistic')
        display(t.groupby(['family', 'statistic']).agg(cells=('outside', 'size'), outside=('outside', 'mean'),
                                                     above=('obs', lambda o: (o > t.hi[o.index]).mean())))

## 2 · Calibration targets (matched by construction; reported, not evidence)

These statistics are compared with 200 draws of each eligible candidate fitted on all rows: the within-day lag 1, the day lag 1, sd_hour, sd_temp and the marginal kurtosis (notebook 17's statistics). Agreement here was built in by the fit, so it does not count as validation (plan §5).

In [4]:
def targets(k):
    proc = samplers.fit_process(E0, day, hour, yr, hour, temp, kind=k)
    st = lambda E: inference.residual_stats(E, day, hour, yr, hour, temp)
    env = inference.envelope(st(E0), [st(proc.simulate(np.random.default_rng(b), day, hour, yr, hour, temp)) for b in range(200)])
    return env.loc[['within_lag1', 'day_lag1', 'sd_hour', 'sd_temp', 'kurt']].groupby(level=0).agg(outside=('outside', 'mean'), median_obs=('obs', 'median'), median_sim=('sim_median', 'median'))

pd.concat({k: targets(k) for k in SUMMARY['eligible']}) if SUMMARY['eligible'] else 'no eligible candidate'

'no eligible candidate'

## 3 · Decision (plan §8)

In [5]:
CH = SUMMARY['chosen']
A_real = sc[CH][1] if CH else np.nan
SUITABLE = bool(CH and SUMMARY['audit_validated'] is True and A_real <= SUMMARY['tau_A'])
Q0 = SUITABLE and SUMMARY['power_ok']
outcome = ('Q0 authorised (exploratory)' if Q0 else 'methods result only: suitable but underpowered' if SUITABLE else 'closed: NEON Q0 development ends for this phase')
print(f"chosen candidate {CH}; audit validated {SUMMARY['audit_validated']}; real-feature A {A_real:.3f} vs τ_A {SUMMARY['tau_A']:.3f} -> suitable {SUITABLE}; "
      f"power ≥ 0.80 at 0.05 nats {SUMMARY['power_ok']} -> {outcome}")

chosen candidate None; audit validated inconclusive (no invalid pair); real-feature A nan vs τ_A 0.562 -> suitable False; power ≥ 0.80 at 0.05 nats False -> closed: NEON Q0 development ends for this phase


## 4 · Exploratory Q0 (only if authorised)

The protocol is that of notebook 18 (context selection per held-out year, V clipped, robust scoring, common support), with the chosen sampler and 199 CRT draws. One p-value; no interval and no confirmatory claim (D-082). The pipeline also computes the cyclic-shift p-value. It is not valid at CPER's persistence (notebooks 11 and 14) and is not reported.

In [6]:
if Q0:
    R1 = R0.assign(d_ref_k=U.D_h.to_numpy(), ref_sensor_id='TRAAT', cond_sensor_id='TRAAT')
    P = suite.pipeline(R1, {'vis': VIS, 'shared_sensor': True, 'n_perm': 199, 'sampler': samplers.factory(CH)})
    print(f"exploratory CRT p = {P['p_crt']:.3f} (199 draws); per-season held-out gains {np.round(P['seas_G'], 3)}")
else: print('Q0 not run: the declared conditions are not met, so no p-value is computed on the real labels')

Q0 not run: the declared conditions are not met, so no p-value is computed on the real labels


## 5 · Reading

**Outcome: closed.** Notebook 19 produced no eligible candidate (plan §4).
- Plan §8 makes this the *closed* outcome, so §1–§2 have nothing to audit and the decision is immediate.
- No p-value was computed on the real labels: the real CPER labels entered notebook 19 only to fit the context-only label generators (D given C, never with V) and the out-of-fold mean M̂(C); no test of the real images against the real labels was run. Here they are used only in §4, which did not run.
- Per D-082 and plan §8, NEON Q0 development closes for this phase with no further design iteration. CPER 2024–2025 stay frozen and unused.

**What stands from the NEON work** (notebooks 07–20):
- the measurement model;
- the verified CPER image set;
- the descriptive per-season gains of notebook 18, which are not a test;
- the methods results of notebooks 11, 14, 17 and 19.

The last of these is that, on CPER-sized single-site data, neither simulation-based nulls nor shift nulls could be certified at the required precision. The question of visual moisture information passes to the field pilot (M2A), whose design must state how its null will be calibrated given this result (see notebook 19 §7).